# D4 · Lab 3 — Few-shot Diversity & Validation-Retry

**Part A proves:** a few-shot set that is all one category biases the classifier; a diverse set fixes it.
**Part B proves:** a validation-retry loop recovers a malformed extraction on attempt 2.

Requires `ANTHROPIC_API_KEY` set and `pip install anthropic pydantic`. Run the cells top to bottom. Full write-up is in `README.md` (this folder).


## Setup


In [ ]:
import os
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY before running (see common/setup-prereq-guide.md)."

from anthropic import Anthropic
import json

client = Anthropic()


### Part A · Biased prompt (5 all-billing examples)
Expect the classifier to over-predict `billing`.


In [ ]:
# PART 1: BIASED prompt — all 5 examples are "billing"
# ─────────────────────────────────────────────
BIASED_SYSTEM = """You are a support ticket classifier. Classify tickets into one of:
billing, technical, account, other
<examples>
<example>
Ticket: I was charged twice last week
Category: billing
</example>
<example>
Ticket: My invoice shows the wrong amount
Category: billing
</example>
<example>
Ticket: I need a refund for my last payment
Category: billing
</example>
<example>
Ticket: My credit card was declined
Category: billing
</example>
<example>
Ticket: Where can I find my receipt?
Category: billing
</example>
</examples>
Respond with ONLY the category word. No explanation."""

MIXED_TICKETS = [
    "I was charged twice this month",           # billing
    "My login isn't working",                    # account
    "The API keeps returning 500 errors",        # technical
    "I need to cancel my subscription",          # account
    "The app crashes on startup",                # technical
    "Invoice sent to wrong email",               # billing
    "Password reset isn't sending emails",       # technical
    "I want to upgrade my plan",                 # account
    "Payment failed but I was still charged",    # billing
    "Two-factor auth codes not arriving",        # technical
    "How do I export my data?",                  # other
    "Account suspended without warning",         # account
    "Refund hasn't arrived after 10 days",       # billing
    "Integration with Slack not working",        # technical
    "I need to add a team member",               # account
    "My free trial ended unexpectedly",          # billing
    "Dashboard loading very slowly",             # technical
    "I forgot which email I used to sign up",    # account
    "Can I use the API for commercial use?",     # other
    "Wrong name on my invoice",                  # billing
]

EXPECTED = [
    "billing","account","technical","account","technical",
    "billing","technical","account","billing","technical",
    "other","account","billing","technical","account",
    "billing","technical","account","other","billing"
]


# Run with biased prompt
biased_results = []
for ticket in MIXED_TICKETS:
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=20,
        system=BIASED_SYSTEM,
        messages=[{"role": "user", "content": f"Ticket: {ticket}"}]
    )
    prediction = response.content[0].text.strip().lower()
    biased_results.append(prediction)

# Score it
biased_correct = sum(p == e for p, e in zip(biased_results, EXPECTED))
print(f"\n=== BIASED PROMPT: {biased_correct}/20 correct ===")
for ticket, pred, exp in zip(MIXED_TICKETS, biased_results, EXPECTED):
    flag = "✅" if pred == exp else "❌"
    print(f"  {flag} [{pred}] (expected: {exp}) — {ticket[:45]}")


### Part A · Diverse prompt (one per category + edge case)
Expect accuracy to recover. Note the before/after improvement.


In [ ]:
# PART 2: DIVERSE prompt — one example per category + edge case
# ─────────────────────────────────────────────
DIVERSE_SYSTEM = """You are a support ticket classifier. Classify tickets into one of:
billing, technical, account, other

<examples>
<example>
Ticket: I was charged twice last week
Category: billing
</example>
<example>
Ticket: The API keeps returning 500 errors and my integration is broken
Category: technical
</example>
<example>
Ticket: I can't log in — my account seems to be locked
Category: account
</example>
<example>
Ticket: Does your service comply with GDPR?
Category: other
</example>
<example>
Ticket: My free trial ended but I never got a charge warning — not sure if this is billing or account
Category: billing
</example>
</examples>

The last example shows an edge case where a ticket could be multiple categories — pick the PRIMARY one.
Respond with ONLY the category word. No explanation."""

# Run with diverse prompt — same 20 tickets
diverse_results = []
for ticket in MIXED_TICKETS:
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=20,
        system=DIVERSE_SYSTEM,
        messages=[{"role": "user", "content": f"Ticket: {ticket}"}]
    )
    prediction = response.content[0].text.strip().lower()
    diverse_results.append(prediction)

diverse_correct = sum(p == e for p, e in zip(diverse_results, EXPECTED))
print(f"\n=== DIVERSE PROMPT: {diverse_correct}/20 correct ===")
for ticket, pred, exp in zip(MIXED_TICKETS, diverse_results, EXPECTED):
    flag = "✅" if pred == exp else "❌"
    print(f"  {flag} [{pred}] (expected: {exp}) — {ticket[:45]}")

print(f"\n📊 Improvement: {biased_correct} → {diverse_correct} (+{diverse_correct - biased_correct})")


## Part B · Validation-retry loop
Business-rule validation (amount > 0, ISO date, non-empty vendor); on failure the errors are sent back for a corrected object.


In [ ]:
from pydantic import BaseModel
from anthropic import Anthropic
from datetime import datetime
import json

client = Anthropic()

# ─────────────────────────────────────────────
# PART 3: Validation-retry loop for invoice extraction
# ─────────────────────────────────────────────
class Invoice(BaseModel):
    vendor: str
    amount: str
    due_date: str

invoice_schema = Invoice.model_json_schema()
invoice_schema["additionalProperties"] = False

def validate_invoice(data: dict) -> list[str]:
    """Returns list of validation errors. Empty = valid."""
    errors = []
    if not data.get("vendor", "").strip():
        errors.append("vendor is empty or missing")
    amount_str = data.get("amount", "")
    try:
        amount_val = float(amount_str.replace("$", "").replace(",", ""))
        if amount_val <= 0:
            errors.append(f"amount must be > 0, got {amount_val}")
    except ValueError:
        errors.append(f"amount '{amount_str}' is not a valid number")
    due_date_str = data.get("due_date", "")
    try:
        datetime.strptime(due_date_str, "%Y-%m-%d")
    except ValueError:
        errors.append(f"due_date '{due_date_str}' is not ISO 8601 (YYYY-MM-DD)")
    return errors

def extract_invoice(invoice_text: str) -> dict:
    """Extract invoice with validation-retry loop. Returns result + attempt count."""
    messages = [
        {
            "role": "user",
            "content": f"Extract vendor, amount, and due_date from this invoice: {invoice_text}"
        }
    ]
    for attempt in range(1, 3):  # max 2 attempts
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=256,
            output_config={
                "format": {"type": "json_schema", "schema": invoice_schema}
            },
            messages=messages
        )
        raw = json.loads(response.content[0].text)
        errors = validate_invoice(raw)
        if not errors:
            return {"data": raw, "attempt": attempt, "success": True}
        # Build follow-up turn with the errors
        messages.append({"role": "assistant", "content": response.content[0].text})
        messages.append({
            "role": "user",
            "content": (
                f"The extracted data failed validation:\n"
                + "\n".join(f"- {e}" for e in errors)
                + "\n\nPlease correct and return valid JSON."
            )
        })
    return {"data": raw, "attempt": 2, "success": False}

# Test cases — some intentionally malformed to trigger retry
TEST_INVOICES = [
    "Invoice from Acme Corp, Due: 2024-01-15, Total: $500.00",
    "Invoice from , Due: Jan 15 2024, Total: -$100",   # 3 failures: empty vendor, bad date, negative amount
    "Received $250 payment from TechCorp on 15/01/2024",  # ambiguous date format
    "Invoice #999 from GlobalSupplies LLC dated 2024-03-01 for $1,200.00",
]


### Part B · Run
One invoice is intentionally malformed (empty vendor, bad date, negative amount) to force the retry. Watch attempt-2 recover it.


In [ ]:
print("\n=== VALIDATION-RETRY LOOP ===")
attempt1_pass = 0
attempt2_recovered = 0

for inv_text in TEST_INVOICES:
    result = extract_invoice(inv_text)
    status = "✅ PASS" if result["success"] else "❌ FAIL"
    attempt_label = f"Attempt {result['attempt']}"
    if result["attempt"] == 1 and result["success"]:
        attempt1_pass += 1
    elif result["attempt"] == 2 and result["success"]:
        attempt2_recovered += 1
    print(f"\n{status} ({attempt_label})")
    print(f"  Input:  {inv_text[:60]}")
    print(f"  Output: {result['data']}")

print(f"\n📊 Attempt-1 successes: {attempt1_pass}/{len(TEST_INVOICES)}")
print(f"📊 Attempt-2 recoveries: {attempt2_recovered}")
